# Binance +200% 暴涨币因子研究

口径：过去 30 天窗口内部，任意较早 4 小时收盘价到严格更晚的 4 小时最高价达到 +200%（价格 3 倍）；不是窗口首尾收益。

In [ ]:
import json
from pathlib import Path
import pandas as pd

REPORT_DIR = Path.cwd()
if not (REPORT_DIR / 'analysis_summary.json').exists():
    REPORT_DIR = Path(r'F:\\9_Crypto\\crypto_trading_system\\reports\\binance_200pct_factor_mining_2026-07-18_final')
summary = json.loads((REPORT_DIR / 'analysis_summary.json').read_text(encoding='utf-8'))
verification = json.loads((REPORT_DIR / 'verification_summary.json').read_text(encoding='utf-8'))
summary['label_definition'], verification['production_comparison']

In [ ]:
events = pd.read_csv(REPORT_DIR / 'current_30d_runups.csv')
event_view = events.assign(
    close_to_high_pct=events['close_to_later_high_return'] * 100,
    low_to_high_pct=events['low_to_later_high_return'] * 100,
    oi_to_mcap_pct=events['oi_to_cmc_mcap'] * 100,
    oi_change_1d_pct=events['oi_change_1d'] * 100,
)[[
    'symbol', 'classification', 'close_to_high_pct', 'low_to_high_pct',
    'duration_days', 'oi_to_mcap_pct', 'oi_change_1d_pct'
]]
assert event_view.query("classification == 'close-confirmed'").shape[0] == 5
assert event_view.query("classification == 'wick-only'").shape[0] == 2
event_view

In [ ]:
metrics = summary['factor_validation']['model_metrics']
simple = metrics['simple_fragility_score']
factor_scorecard = pd.DataFrame([
    {
        'sample': sample,
        'rows': simple[sample]['n'],
        'positives': simple[sample]['positives'],
        'base_rate_pct': simple[sample]['base_rate'] * 100,
        'auc': simple[sample]['auc'],
        'top_2pct_precision_pct': simple[sample]['top_2pct_precision'] * 100,
        'top_2pct_lift': simple[sample]['top_2pct_lift'],
    }
    for sample in ['train', 'test']
])
factor_scorecard

In [ ]:
factor_stability = pd.read_csv(REPORT_DIR / 'factor_stability.csv')
stable_factors = factor_stability.query('stable').sort_values('test_auc', ascending=False)
stable_factors[[
    'factor', 'direction', 'train_auc', 'test_auc',
    'train_top_decile_lift', 'test_top_decile_lift'
]].head(12)

In [ ]:
test_strategy = pd.read_csv(REPORT_DIR / 'strategy_backtest_summary.csv')
train_strategy = pd.read_csv(REPORT_DIR / 'strategy_train_summary.csv')
strategy_comparison = pd.concat([
    train_strategy.assign(sample='train'),
    test_strategy.assign(sample='test'),
]).query("strategy == 'simple_fragility_score' and top_k == 3")
strategy_comparison[[
    'sample', 'top_k', 'median_phase_mean_net_return',
    'median_phase_median_net_return', 'median_phase_win_rate',
    'median_phase_hit_target_rate', 'worst_trade_drawdown'
]]

## 结论

`fragility_score = mean(percentile(log_amihud_7d), percentile(intraday_range_1d), percentile(atr_7d_pct))`。它适合形成每日预警池，不足以直接证明可交易利润。OI 仅作为拥挤度过滤，因为历史 OI 覆盖不足，且当前主样本并未一致显示 OI 扩张。